In [1]:
import torch, torch.nn as nn, time, json
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [2]:
tf_ = transforms.ToTensor()
train_ds = datasets.FashionMNIST("data", train=True, download=True, transform=tf_)
test_ds = datasets.FashionMNIST("data", train=False, download=True, transform=tf_)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)
print("Train images:", len(train_ds), "| Test images:", len(test_ds))

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 210kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.92MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.8MB/s]

Train images: 60000 | Test images: 10000


In [3]:
pt_model = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 10),
).to(device)

opt = torch.optim.Adam(pt_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

start = time.time()
for epoch in range(8):
    pt_model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(pt_model(x), y)
        loss.backward()
        opt.step()
    print(f"Epoch {epoch+1}/8 done, loss {loss.item():.4f}")
pt_time = time.time() - start
print(f"Training time: {pt_time:.0f}s")

Epoch 1/8 done, loss 0.4594
Epoch 2/8 done, loss 0.2605
Epoch 3/8 done, loss 0.2504
Epoch 4/8 done, loss 0.2606
Epoch 5/8 done, loss 0.3412
Epoch 6/8 done, loss 0.3099
Epoch 7/8 done, loss 0.2217
Epoch 8/8 done, loss 0.2246
Training time: 59s


In [4]:
pt_model.eval()
correct = 0
with torch.no_grad():
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        correct += (pt_model(x).argmax(1) == y).sum().item()
pt_acc = correct / len(test_ds)
print(f"PyTorch test accuracy: {pt_acc:.4f}")
torch.save(pt_model.state_dict(), "pytorch_cnn.pt")

PyTorch test accuracy: 0.9102


In [5]:
import tensorflow as tf, numpy as np

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = (x_train / 255.0).astype("float32")[..., np.newaxis]
x_test = (x_test / 255.0).astype("float32")[..., np.newaxis]

tf_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10),
])
tf_model.compile(optimizer="adam",
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=["accuracy"])

start = time.time()
tf_model.fit(x_train, y_train, epochs=8, batch_size=128, validation_split=0.1)
tf_time = time.time() - start

_, tf_acc = tf_model.evaluate(x_test, y_test)
print(f"TensorFlow test accuracy: {tf_acc:.4f}")
tf_model.save("tf_cnn.keras")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.8062 - loss: 0.5363 - val_accuracy: 0.8647 - val_loss: 0.3529
Epoch 2/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8760 - loss: 0.3467 - val_accuracy: 0.8928 - val_loss: 0.2947
Epoch 3/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8933 - loss: 0.2948 - val_accuracy: 0.8977 - val_loss: 0.2681
Epoch 4/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9029 - loss: 0.2668 - val_accuracy: 0.9033 - val_loss: 0.2472
Epoch 5/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9113 - loss: 0.2428 - val_accuracy: 0.9092 - val_loss: 0.2389
Epoch 6/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9184 - loss: 0.2222 - val_accuracy: 0.9142 - val_loss: 0.2331
Epoch 7/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/st

In [6]:
print("| Framework  | Test accuracy | Training time |")
print("|------------|---------------|---------------|")
print(f"| PyTorch    | {pt_acc:.4f}        | {pt_time:.0f}s          |")
print(f"| TensorFlow | {tf_acc:.4f}        | {tf_time:.0f}s          |")

| Framework  | Test accuracy | Training time |
|------------|---------------|---------------|
| PyTorch    | 0.9102        | 59s          |
| TensorFlow | 0.9140        | 29s          |


In [7]:
!pip install -q fastapi httpx python-multipart

In [8]:
import io
from fastapi import FastAPI, UploadFile, File
from fastapi.testclient import TestClient
from PIL import Image

CLASSES = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Ankle boot"]

app = FastAPI(title="Fashion Classifier API")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    img = Image.open(io.BytesIO(await file.read())).convert("L").resize((28, 28))
    x = torch.tensor(np.array(img, dtype="float32") / 255.0).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(pt_model(x), dim=1)[0]
    i = int(probs.argmax())
    return {"prediction": CLASSES[i], "confidence": round(float(probs[i]), 4)}

client = TestClient(app)
print(client.get("/health").json())

img = Image.fromarray((test_ds[0][0][0].numpy() * 255).astype("uint8"))
buf = io.BytesIO(); img.save(buf, format="PNG"); buf.seek(0)
r = client.post("/predict", files={"file": ("a.png", buf, "image/png")})
print("API says:", r.json(), "| True label:", CLASSES[test_ds[0][1]])

{'status': 'ok'}
API says: {'prediction': 'Ankle boot', 'confidence': 0.9993} | True label: Ankle boot
